<a href="https://colab.research.google.com/github/sharad-25/fpl-data/blob/main/FPL_Model_Weekly_LATEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FPL Transfer Model — weekly run

Structural expected-points engine → Monte Carlo → multi-period MILP → Google Sheet.

**Run order matters.** Snapshots are written first because `bootstrap-static` is a
view of *now*: if you don't save it this week, this week's training data is gone
permanently.

Run every cell top to bottom after each gameweek deadline.

## 1. Setup

**Edit one line below: `PKG_DIR`.** Point it at the Drive folder holding
`fplmodel.zip` (or an already-unzipped `fplmodel/`). Everything else finds itself.

In [13]:
!pip install -q pulp gspread gspread-formatting pyarrow

import os, sys

# ---- the only line you need to edit -------------------------------------
PKG_DIR = '/content/drive/MyDrive/FPL'
# -------------------------------------------------------------------------

# Drive is mounted FIRST. Config creates its data_dir on construction, and a
# stray folder sitting at the mount point stops Drive mounting over it.
_mounted = os.path.isdir('/content/drive/MyDrive')
try:
    from google.colab import drive
    if not _mounted:
        drive.mount('/content/drive')
    _mounted = os.path.isdir('/content/drive/MyDrive')
except ImportError:
    _mounted = None          # not on Colab at all; PKG_DIR is a local path
except Exception as e:
    print('Drive mount raised:', e)

# A failed mount used to be swallowed and printed as a friendly note, which is
# the worst thing it could do: everything downstream -- the settings file, the
# package zip, the snapshots, the projection cache -- lives in Drive, so the
# run would either die twenty lines later with a confusing "could not find
# fplmodel", or, worse, quietly succeed against a STALE copy left in /content
# by a previous session and write last week's numbers into the sheet.
#
# Google's OAuth popup fails often enough to deserve this: a 401 "malformed
# request" from accounts.google.com is usually stale runtime auth state or
# several signed-in Google accounts, not anything wrong here.
if _mounted is False or (_mounted and not os.path.isdir(PKG_DIR)):
    raise SystemExit(
        'DRIVE IS NOT AVAILABLE -- stopping before anything reads a stale copy.\n\n'
        + ('  The mount did not complete.\n' if _mounted is False else
           '  Drive mounted, but PKG_DIR does not exist: ' + str(PKG_DIR) + '\n')
        + '\n  If Google showed a 401 "malformed request", try in this order:\n'
          '    1. Runtime > Disconnect and delete runtime, reconnect, re-run.\n'
          '    2. Open Colab in incognito, signed into ONE Google account.\n'
          '    3. Allow third-party cookies for accounts.google.com.\n'
          '\n  Or run without Drive: drag fplmodel.zip and fpl_settings.json\n'
          '  into the Colab file browser and set PKG_DIR = "/content".\n'
          '  Snapshots and the projection cache will not persist, but the\n'
          '  run works and the Sheet output is unaffected.\n')

# Find fplmodel/, unzipping fplmodel.zip if that is all there is.
#
# This used to compare mtimes to decide whether to re-extract, which is the
# wrong question and answered it wrongly: `unzip -o` stamps the extracted
# directory with the extraction time, so after one run the unzipped copy is
# always "newer" than the zip and a genuinely newer zip is never extracted.
# The right question is which VERSION each one holds, and the zip can be
# asked that without extracting it.
_QUOTES = chr(34) + chr(39)


def _version_in(src):
    for ln in src.splitlines():
        if ln.replace(' ', '').startswith('__version__='):
            return ln.split('=', 1)[1].strip().strip(_QUOTES)
    return None


def _zip_version(z):
    try:
        import zipfile
        with zipfile.ZipFile(z) as f:
            for n in f.namelist():
                if n.endswith('fplmodel/__init__.py'):
                    return _version_in(f.read(n).decode('utf-8', 'replace'))
    except Exception:
        return None
    return None


def _dir_version(d):
    try:
        with open(os.path.join(d, 'fplmodel', '__init__.py')) as f:
            return _version_in(f.read())
    except Exception:
        return None


def _locate_package():
    for d in (PKG_DIR, '/content', os.getcwd()):
        if not d or not os.path.isdir(d):
            continue
        pkg, z = os.path.join(d, 'fplmodel'), os.path.join(d, 'fplmodel.zip')
        if os.path.isfile(z):
            zv, dv = _zip_version(z), _dir_version('/content')
            if zv is None or zv != dv:
                # Wipe first. `unzip -o` overwrites but never DELETES, so a
                # module removed in the new release lingers and keeps being
                # importable -- a stale copy that no longer exists upstream is
                # the worst kind, because nothing reports it.
                os.system('rm -rf /content/fplmodel')
                os.system(f'unzip -q -o "{z}" -d /content')
                if zv:
                    print(f'extracted fplmodel {zv} from {z}'
                          + (f' (replacing {dv})' if dv else ''))
            if os.path.isdir('/content/fplmodel'):
                return '/content'
            # The archive extracted but the package is not where the import
            # needs it. This happened for real: two releases were built by
            # zipping the PROJECT FOLDER from its parent, so everything landed
            # one level down as `fpl_model/fplmodel/...`. The version probe
            # above matches any path ENDING in `fplmodel/__init__.py`, so it
            # read the version happily and then the import failed -- printing
            # "extracted fplmodel 1.54.0" immediately followed by "Could not
            # find fplmodel", which reads as a missing file rather than a bad
            # archive. Say what is actually wrong.
            #
            # No backslash escapes anywhere in here. This cell is generated
            # from a plain triple-quoted string in build_notebook.py, so a
            # newline escape written here is interpreted when THAT file is
            # parsed and lands in the notebook as a real line break inside a
            # string literal -- which is a syntax error in cell 2 and takes
            # the whole run down. Same reason the quote characters above are
            # built with chr(). Lines and a runtime join, instead.
            import glob as _g
            _nested = _g.glob('/content/*/fplmodel/__init__.py')
            if _nested:
                _wrap = _nested[0].split('/content/')[1].split('/')[0]
                raise SystemExit(chr(10).join([
                    'THE ZIP HAS THE WRONG LAYOUT -- not a missing file.',
                    '',
                    '  ' + str(z),
                    '  holds the package one level down, under ' + _wrap
                    + '/, so /content/fplmodel',
                    '  was never created and the import cannot work.',
                    '',
                    '  The package directory has to sit at the ZIP ROOT. Ask',
                    '  for an archive built by build_release.py, which refuses',
                    '  to produce this layout, and REPLACE the file in your',
                    '  Drive folder.',
                    '',
                    '  Then check that the version on the line above CHANGES.',
                    '  A zip still reporting the old version was never',
                    '  swapped, and that is a different problem with the same',
                    '  symptom.',
                ]))
        if os.path.isdir(pkg):
            return d
    return None

root = _locate_package()

# Re-running this cell after dropping in a new zip must not keep last run's
# modules alive. Python caches by name, so without this the import below is a
# no-op and the version printed is the one already in memory.
for _m in [m for m in sys.modules if m == 'fplmodel' or m.startswith('fplmodel.')]:
    del sys.modules[_m]

# The zip carries the notebook as well as the package, and the notebook is the
# half people forget to replace -- the Live tab is a notebook cell, so a stale
# notebook silently produces a workbook with no Live tab. Drop the fresh copy
# beside the settings file where it is easy to find.
try:
    import shutil
    _fresh = os.path.join(root or '/content', 'FPL_Model_Weekly.ipynb')
    if os.path.isfile(_fresh) and os.path.isdir(PKG_DIR):
        _dest = os.path.join(PKG_DIR, 'FPL_Model_Weekly_LATEST.ipynb')
        if (not os.path.exists(_dest)
                or os.path.getmtime(_fresh) > os.path.getmtime(_dest)):
            shutil.copyfile(_fresh, _dest)
            print('a newer notebook is in your Drive folder as '
                  'FPL_Model_Weekly_LATEST.ipynb -- open it if the version '
                  'warning below fires')
except Exception:
    pass

if root is None:
    raise SystemExit(
        'Could not find fplmodel. Put fplmodel.zip (or the unzipped fplmodel/ '
        f'folder) in {PKG_DIR}, then re-run this cell.')

sys.path.insert(0, root)
from fplmodel.config import Config
from fplmodel.pipeline import run_weekly, optimise_from_board
from fplmodel import sheet as sht, validate as val, optimise as opt, simulate as sm
import pandas as pd, numpy as np
import fplmodel
pd.set_option('display.width', 220, 'display.max_columns', 60)

print(f'fplmodel {fplmodel.__version__} loaded from {root}')
print(' ', fplmodel.__changes__.get(fplmodel.__version__, ''))

# This notebook and the package ship together in the same zip, and Colab keeps
# whatever notebook you opened. Both halves therefore go stale independently,
# and BOTH directions have now cost a run:
#
#   old notebook + new package -> a renamed column, bare KeyError twelve cells
#                                 in, mid-gameweek
#   new notebook + old package -> TypeError: refresh_live() got an unexpected
#                                 keyword argument 'league_id'
#
# The old guard missed the second one twice over. It compared only major.minor,
# so 1.17.1's notebook against 1.17.0's package looked like a match; and it
# printed rather than stopped, so a mismatch scrolled past in cell 1 and the
# failure surfaced later as something that looked like a code bug. Exact
# version, and a hard stop.
NOTEBOOK_FOR = '1.57.0'
if fplmodel.__version__ != NOTEBOOK_FOR:
    # Tuples, not strings: '1.9.0' < '1.17.0' is False lexicographically, and
    # getting this backwards would tell you to replace the half that is fine.
    def _v(s):
        return tuple(int(x) if x.isdigit() else 0 for x in s.split('.'))
    _newer = _v(fplmodel.__version__) < _v(NOTEBOOK_FOR)
    raise SystemExit(
        'VERSION MISMATCH -- stopping here rather than failing later.\n'
        '  this notebook was built for fplmodel ' + NOTEBOOK_FOR + '\n'
        '  the package that loaded is      fplmodel ' + fplmodel.__version__
        + ' (from ' + str(root) + ')\n\n'
        + ('  The PACKAGE is the stale half. Put the newest fplmodel.zip in\n'
           '  ' + str(PKG_DIR) + ' and re-run this cell -- it re-extracts\n'
           '  automatically when the zip holds a different version.\n'
           if _newer else
           '  The NOTEBOOK is the stale half. Open FPL_Model_Weekly_LATEST.ipynb\n'
           '  from ' + str(PKG_DIR) + ' -- cell 1 above just copied it there\n'
           '  out of the zip. Your settings are in fpl_settings.json, so\n'
           '  swapping notebooks loses nothing.\n'))

# Your team id and league id live in fpl_settings.json in your Drive folder,
# NOT in this notebook -- so replacing the notebook can never wipe them.
from fplmodel import settings as st
S = st.load(PKG_DIR)
print()
print(st.describe(S, PKG_DIR))
# If this version is not the one you just downloaded, Colab is running a stale
# copy: Runtime > Restart session, then run this cell again.

fplmodel 1.57.0 loaded from /content
  THE BPS PATH, AUDITED END TO END. Three defects, all measured. (1) THE GOAL MAP WAS INVERTED. BPS_GOAL held GK/DEF 24 and FWD 12; FPL pays the opposite -- GK/DEF 12, MID 18, FWD 24. Verified twice without assuming the rule: the within-player BPS uplift for exactly one goal (same player, no assist, 60+ minutes, no card) is DEF 16.0 / MID 21.0 / FWD 25.4 across 2025/26 and DEF 16.5 / MID 21.6 / FWD 29.0 across 2026/27 GW1-5, each a few BPS above the rule because a goal drags shot and big-chance credit with it. Defenders were credited double for a goal and forwards half, which is the main reason the walk-forward showed defender bonus 44% over and forward bonus 21% under. (2) THE BASE DOUBLE-COUNTED FOUR STREAMS. `bps_base` stripped goals, assists and clean sheets; the simulation then added appearance, saves, DefCon units and yellows on top of a base that still contained them. Over all 60+ minute matches of 2025/26 the double count was +11.3 BPS a mat

## 2. Configuration

`objective='points'` maximises raw expected points, which is right for a mini-league.
Set `eo_weight` above 0 only if you start chasing overall rank — it discounts
heavily-owned players so differentials rank higher.

In [14]:
cfg = Config(
    season           = S.season,
    data_dir         = S.data_dir,
    short_horizon    = S.short_horizon,
    long_horizon     = S.long_horizon,
    optimiser_horizon= S.long_horizon,
    future_decay     = 0.84,        # weight on each later gameweek in the MILP
    n_sims           = S.n_sims,
    sheet_name       = S.sheet_name,
    odds_api_key     = S.odds_api_key,
    use_player_props = S.use_player_props,
    prop_weight      = S.prop_weight,
    prop_share_alpha = S.prop_share_alpha,
    goal_budget_blend= S.goal_budget_blend,
    dc_promoted_att_offset = S.dc_promoted_att_offset,
    dc_promoted_def_offset = S.dc_promoted_def_offset,
    clubelo_manual   = S.clubelo_manual,
    clubelo_asof     = S.clubelo_asof,
)
print('data dir:', cfg.data_dir)
if not S.ready:
    raise SystemExit('Set entry_id in fpl_settings.json (Drive), then re-run cell 1.')

data dir: /content/drive/MyDrive/fpl_data


## 3. Run the model

Pulls the FPL API, fits Dixon-Coles on this season's results (Elo-seeded), prices
GW+1 from bookmaker odds where available, models minutes, shrinks every rate, then
simulates.

In [15]:
board, extras = run_weekly(cfg)
next_gw = extras['next_gw']
print(f"\n{len(board)} players scored for GW{next_gw}")

next gameweek: 6
snapshotted 667 players, 3216 history rows
GW6: skipped -- GW6 is not settled (upcoming). Bonus and the dubious goals panel can still move points, so scoring now would write numbers that later change. Pass force=True to override.
clubelo: 20/20 teams from ratings entered by hand (as of 2026-09-17); the network is not consulted
prior: fitted on 272 matches from 2526
understat: team xG for 30/50 played fixtures (the rest keep FPL's)
results: 50 matches this season (50 with xG, 50 blended with closing odds)
codes: 2025-26 resolved 100% of rows to a stable player code
continuity: measured for 17 clubs, lowest Spurs 0.54 (mean 0.72); 3 promoted clubs held at 0.5x, marked down +0.148 attack / +0.165 defence
dixon-coles: 50 matches this season, fitted on xg, prior=last-season, prior weight 9, home advantage 0.249
odds: 20 fixtures from the-odds-api
projected 120 team-fixtures (40 priced from odds; 20/20 of GW6)
priors: last-season rates for 481 players; own-history weight med

### What the shrinkage constants are telling you

`k` is the number of minutes at which a player's own record and the prior carry
equal weight. Two gameweeks is ~180 minutes, so anything with `k` above ~700 is
still almost entirely last season's number — which is correct, not a bug.

In [16]:
mins_played = 180
for stat, k in extras['shrinkage_k'].items():
    own = mins_played / (mins_played + k)
    print(f'{stat:16s} k={k:7.0f}   at 180 mins: {own:4.0%} this season, {1-own:4.0%} prior')

xa_per90         k=    400   at 180 mins:  31% this season,  69% prior
saves_per90      k=   2000   at 180 mins:   8% this season,  92% prior
defcon_per90     k=    715   at 180 mins:  20% this season,  80% prior
tackles_per90    k=    715   at 180 mins:  20% this season,  80% prior
cbi_per90        k=    715   at 180 mins:  20% this season,  80% prior
recov_per90      k=    715   at 180 mins:  20% this season,  80% prior
bps_per90        k=    868   at 180 mins:  17% this season,  83% prior
yellow_per90     k=   1550   at 180 mins:  10% this season,  90% prior
threat_per90     k=    298   at 180 mins:  38% this season,  62% prior
creativity_per90 k=    270   at 180 mins:  40% this season,  60% prior
xg_per90_direct  k=    700   at 180 mins:  20% this season,  80% prior


## 4. The board

In [17]:
from fplmodel.naming import label as _label

def show(df, cols, n=None, by=None):
    """Display `cols` under their sheet headers, skipping any that are absent.

    A hardcoded column list is a trap in a notebook you re-run every week: the
    package moves, a column is renamed, and a cell twelve steps into a
    twenty-minute run dies on a KeyError. That happened -- 'P10'/'P90' became
    Floor/Ceiling and this cell stopped a live gameweek run dead. Now a
    missing column is reported and the rest still prints.
    """
    have = [c for c in cols if c in df.columns]
    gone = [c for c in cols if c not in df.columns]
    if gone:
        print(f'note: not in this version of the board, skipped: {gone}')
    out = df if by is None else df.nlargest(n, by) if n else df.sort_values(by)
    return out[have].rename(columns={c: _label(c) for c in have})

# Board keys, not sheet headers -- the frame keeps its internal names and
# fplmodel.naming maps them to what the spreadsheet shows.
cols = ['Player','Team','£','STS','LTS','xP2','xP6','Floor','Ceiling','Val',
        'xMin','Fit','Conf','Next 6','AttE6','CSE6','Sw','Luck','DC%','pHaul']
show(board[board.position=='MID'], cols, n=15, by='LTS')

,Player,Team,Price,Short Score,Long Score,xP Next 2,xP Next 6,Floor,Ceiling,xP per £m,Mins Next GW,Available,Mins Played,Next 6 Fixtures,Att Ease 6,xCS6,Fixture Swing,Luck (pts),P(DefCon),P(10+ pts)
517,B.Fernandes,MUN,11.9,100.0,100.0,12.97,38.96,24.40,54.43,3.49,86.0,100%,450.0,TOT(H) LEE(A) BOU(H) CHE(A) AVL(H) LIV(A),7.66,1.68,0.37,4.4,0.055617,0.320
518,Mbeumo,MUN,7.9,99.0,100.0,10.25,30.82,18.76,45.02,4.16,85.0,100%,450.0,TOT(H) LEE(A) BOU(H) CHE(A) AVL(H) LIV(A),7.66,1.68,0.37,-0.6,0.016426,0.208
11,Saka,ARS,9.6,100.0,99.0,11.75,36.00,22.16,51.07,3.89,76.0,100%,416.0,LEE(H) NFO(A) EVE(H) LIV(A) HUL(H) NEW(A),8.07,2.85,0.21,-2.5,0.052201,0.221
136,Groß,BHA,5.9,99.0,99.0,9.26,25.67,15.81,37.20,4.68,84.0,100%,450.0,SUN(A) CRY(H) LIV(A) MCI(A) BRE(H) HUL(A),6.36,1.54,0.40,25.8,0.094818,0.116
582,Gibbs-White,NFO,8.0,95.0,99.0,7.88,25.09,15.14,36.90,3.31,86.0,100%,450.0,CRY(A) ARS(H) IPS(A) BRE(A) MCI(H) BOU(A),5.09,1.48,0.00,5.3,0.015612,0.136
279,Dewsbury-Hall,EVE,6.6,96.0,98.0,8.07,23.26,14.71,33.10,3.83,86.0,100%,450.0,HUL(A) CHE(H) ARS(A) NEW(A) COV(H) BRE(A),5.20,1.60,-0.28,-6.7,0.146081,0.113
544,Barnes,NEW,6.1,98.0,98.0,8.91,23.96,14.19,35.01,4.15,86.0,100%,450.0,COV(A) AVL(H) CRY(A) EVE(H) FUL(A) ARS(H),5.55,1.46,0.88,13.0,0.039976,0.134
653,Xhaka,SUN,5.5,94.0,98.0,7.49,22.79,15.14,31.22,4.38,85.0,100%,450.0,BHA(H) BOU(A) LEE(H) COV(A) CHE(H) AVL(A),5.10,1.68,0.13,-0.9,0.319692,0.050
446,Wirtz,LIV,7.3,97.0,97.0,8.52,25.07,14.86,36.22,3.70,78.0,100%,402.0,MCI(H) BRE(A) BHA(H) ARS(H) CRY(A) MUN(H),6.28,1.45,0.52,1.6,0.101424,0.105
448,Szoboszlai,LIV,6.9,97.0,97.0,8.26,24.22,14.87,35.31,3.78,80.0,100%,440.0,MCI(H) BRE(A) BHA(H) ARS(H) CRY(A) MUN(H),6.28,1.45,0.52,-2.1,0.142151,0.094


## 5. Your squad

Nothing to type. Put your team id in and it imports the squad, your bank, your
free transfers and your chips. Buy prices are reconstructed too: anyone you
transferred in has his price in the transfers feed, and anyone you never
transferred came from your initial squad, so his price is his GW1 value.

Your team id is in the URL when you open your team on the FPL site:
`.../entry/`**`1234567`**`/event/2`

In [18]:
from fplmodel import myteam as mtm

MY_ENTRY_ID = S.entry_id        # from fpl_settings.json -- nothing to edit here

if MY_ENTRY_ID:
    team = mtm.load_team(MY_ENTRY_ID, next_gw, cfg, fpl=extras['fpl'])
    print(team.summary())

    squad_ids      = team.squad_ids
    buy_prices     = team.buy_prices
    buy_gws        = team.buy_gws
    BANK           = team.bank
    FREE_TRANSFERS = team.free_transfers
    CHIPS_LEFT     = ', '.join(mtm.remaining_chips(team.chips_used))
else:
    # Manual fallback -- only needed before the first deadline of the season,
    # when picks are not yet public.
    MY_SQUAD = {}          # {player_id: (buy_price_or_None, buy_gw)}
    BANK, FREE_TRANSFERS, CHIPS_LEFT = 0.0, 1, 'TC, WC, FH'
    squad_ids  = list(MY_SQUAD)
    buy_prices = {int(k): v[0] for k, v in MY_SQUAD.items()}
    buy_gws    = {int(k): v[1] for k, v in MY_SQUAD.items()}
    print('Set MY_ENTRY_ID to import your team automatically.')

# The most you are ever willing to pay for an extra transfer. 4 means one hit;
# 0 refuses them. The planner never proposes more than this.
MAX_HIT = 4

if not squad_ids:
    raise SystemExit(
        'No squad loaded. Set entry_id in fpl_settings.json and re-run cell 1. '
        'Continuing would write an empty My Team tab with no error.')

display(show(board[board.id.isin(squad_ids)]
             .sort_values(['position','LTS'], ascending=[True, False]),
             ['position'] + cols))

squad: 15 players (as of GW5)
bank: 0.0   squad value: 100.6
free transfers: 3
chips used: bboost (GW2), wildcard (GW4)
buy prices recovered: 15/15


,position,Player,Team,Price,Short Score,Long Score,xP Next 2,xP Next 6,Floor,Ceiling,xP per £m,Mins Next GW,Available,Mins Played,Next 6 Fixtures,Att Ease 6,xCS6,Fixture Swing,Luck (pts),P(DefCon),P(10+ pts)
7,DEF,Calafiori,ARS,5.9,100.0,100.0,9.70,30.30,19.03,43.06,5.13,80.0,100%,416.0,LEE(H) NFO(A) EVE(H) LIV(A) HUL(H) NEW(A),8.07,2.85,0.03,-1.1,0.090233,0.091
474,DEF,Gvardiol,MCI,5.7,99.0,99.0,8.59,25.41,14.48,37.65,4.62,84.0,100%,434.0,LIV(A) IPS(H) AVL(A) BHA(H) NFO(A) FUL(H),8.03,2.06,-0.13,13.1,0.119535,0.058
127,DEF,De Cuyper,BHA,5.0,98.0,93.0,8.37,22.52,12.09,34.42,4.84,77.0,100%,423.0,SUN(A) CRY(H) LIV(A) MCI(A) BRE(H) HUL(A),6.36,1.54,0.04,8.7,0.076470,0.089
199,DEF,Thomas,COV,4.0,82.0,84.0,5.11,17.67,10.27,26.07,5.59,81.0,100%,437.0,NEW(H) TOT(A) FUL(H) SUN(H) EVE(A) CRY(H),4.60,1.40,-0.22,3.9,0.335699,0.052
26,DEF,Konsa,ARS,4.6,82.0,83.0,5.89,18.55,9.16,29.51,3.96,56.0,75%,281.0,LEE(H) NFO(A) EVE(H) LIV(A) HUL(H) NEW(A),8.07,2.85,0.03,-0.9,0.107138,0.039
491,FWD,Haaland,MCI,15.6,100.0,100.0,13.41,37.55,22.05,54.13,2.40,86.0,100%,450.0,LIV(A) IPS(H) AVL(A) BHA(H) NFO(A) FUL(H),8.03,2.06,0.25,12.8,0.000524,0.156
289,FWD,Barry,EVE,5.7,96.0,94.0,7.86,22.05,11.80,32.55,3.93,80.0,100%,417.0,HUL(A) CHE(H) ARS(A) NEW(A) COV(H) BRE(A),5.20,1.60,-0.28,-8.4,0.004359,0.072
184,FWD,João Pedro,CHE,7.7,82.0,85.0,6.54,17.94,6.10,31.51,2.29,50.0,75%,360.0,BOU(H) EVE(A) TOT(H) MUN(H) SUN(A) LEE(H),6.51,1.61,0.91,9.4,0.002008,0.104
123,GK,Verbruggen,BHA,4.5,97.0,95.0,7.19,20.76,12.99,28.98,4.62,87.0,100%,450.0,SUN(A) CRY(H) LIV(A) MCI(A) BRE(H) HUL(A),6.36,1.54,0.04,6.0,0.000000,0.023
604,GK,Kinsky,TOT,4.5,86.0,86.0,6.12,18.84,11.14,27.33,4.13,79.0,100%,450.0,MUN(A) COV(H) CHE(A) CRY(H) LEE(A) IPS(H),6.18,1.61,-0.21,1.9,0.000000,0.014


## 6. Mini-league

Global ownership is the wrong denominator when you're racing eleven people you know.
A player owned by 4% of the world but by six of your ten rivals is *template to you*.

Find your league id in the URL when you open the league on the FPL site:
`.../leagues/`**`1234567`**`/standings/c`

In [19]:
from fplmodel import league as lg, rank as rk

LEAGUE_ID = S.league_id         # from fpl_settings.json
GWS_LEFT = 38 - next_gw + 1     # gameweeks left including this one

if LEAGUE_ID and MY_ENTRY_ID:
    # Which gameweek's squads describe the league RIGHT NOW?
    #
    # Before a deadline: the last completed gameweek, and nothing better
    # exists. Rivals' picks are not public until the deadline, and last
    # week's team is a guess at this week's -- a decent guess, but a guess.
    #
    # After it: THIS gameweek, and the difference is not incremental. Every
    # eleven, every armband and every chip in the league is a published fact
    # that nobody can change. Reading last week's squads in that window threw
    # away the single best information state of the entire week -- and it is
    # the window a manager is actually sitting in when he opens the sheet,
    # because that is when the deadline has just made him think about it.
    #
    # Chips are the sharpest case. A rival on Bench Boost is fielding fifteen
    # players and one on Triple Captain has a x3 on somebody; neither is
    # visible in last week's picks, and both change a head-to-head by six to
    # thirteen points. Guessing cannot recover them. Reading them is free.
    LOCKED = extras['fpl'].picks_public(next_gw)
    ml_gw = next_gw if LOCKED else next_gw - 1
    standings, picks = lg.fetch_league_picks(LEAGUE_ID, ml_gw, cfg, max_entries=60)
    eo = lg.effective_ownership(picks, exclude_entry=MY_ENTRY_ID)
    board = lg.add_league_columns(board, eo, squad_ids=squad_ids)
    ctx = lg.league_context(standings, MY_ENTRY_ID)

    # --- expected rank change -------------------------------------------
    rivals   = rk.build_rival_squads(picks, exclude_entry=MY_ENTRY_ID)
    totals   = dict(zip(standings.entry_id.astype(int),
                        pd.to_numeric(standings.total, errors='coerce').fillna(0)))
    posmap   = dict(zip(board.id.astype(int), board.element_type.astype(int)))
    xminsmap = dict(zip(board.id.astype(int), board.xMin.fillna(0)))
    rank_gws = list(range(next_gw, next_gw + cfg.long_horizon))

    # `lineups_locked` pins the first gameweek of the horizon to each rival's
    # DECLARED eleven and armband instead of assuming he fields his best XI.
    # Correct only once the deadline has gone -- which is exactly what LOCKED
    # says. LeagueSim has carried this argument since the rank sim was
    # written and nothing ever passed it, because `ml_gw = next_gw - 1` meant
    # the picks on hand were always last week's and pinning them would have
    # frozen every rival into a team he was about to change.
    ls = rk.LeagueSim(extras['sim'], rank_gws, rivals, totals,
                      my_entry=MY_ENTRY_ID, my_squad=squad_ids, positions=posmap,
                      residual_gws=max(0, GWS_LEFT - cfg.long_horizon),
                      residual_sd_per_gw=12.0,
                      lineups_locked=LOCKED)

    total_arr, idx = rk.horizon_totals(extras['sim'], rank_gws)
    repl   = rk.replacement_vectors(total_arr, idx, posmap, xminsmap)
    impact = rk.rank_impact(ls, board.id.astype(int).tolist(), repl)
    board  = rk.add_rank_columns(board, impact)
    board['MLFlag'] = rk.chase_flag(board, squad_ids, stance=ctx.get('stance','chase'))

    outlook = ls.summary()
    print(f"{len(standings)} teams, picks as of GW{ml_gw}"
          + (" -- LOCKED: every rival's XI, armband and chip is final"
             if LOCKED else " -- last completed gameweek; this week's squads "
             "are not public yet"))
    if LOCKED:
        _chips = {e: (i.get('chip') or '-') for e, i in rivals.items()}
        _active = {e: c for e, c in _chips.items() if c not in ('-', None, '')}
        if _active:
            _nm = dict(zip(standings.entry_id.astype(int), standings.manager))
            print("  chips active this gameweek: "
                  + ", ".join(f"{_nm.get(e, e)} ({c})"
                              for e, c in sorted(_active.items(),
                                                 key=lambda kv: str(kv[1]))))
    print(f"you are {ctx.get('my_rank')} of {ctx.get('n_teams')}, "
          f"{ctx.get('points_behind_leader', 0):.0f} behind")
    print(f"projected finish {outlook['expected_rank']:.2f} "
          f"(best case {outlook['rank_p10']:.0f}, worst case "
          f"{outlook['rank_p90']:.0f}), "
          f"P(win) {outlook['p_win']:.1%}, P(top 3) {outlook['p_top3']:.1%}")
    print(ctx.get('read', ''))
else:
    ctx, standings, eo, ls, outlook = {}, None, None, None, {}
    board = lg.add_league_columns(board, None, squad_ids=squad_ids)
    board = rk.add_rank_columns(board, None)
    print('No LEAGUE_ID set - ML columns left blank.')

20 teams, picks as of GW5 -- last completed gameweek; this week's squads are not public yet
you are 17 of 20, 67 behind
projected finish 11.97 (best case 4, worst case 19), P(win) 2.6%, P(top 3) 8.9%
67 points behind. Matching the field cannot close that. Read MLD as the ranking it was built for.


### Where you actually stand

`P(above)` is the probability you finish ahead of each manager, from the simulation
rather than the current gap. A 40-point deficit to someone with a thin squad is a
different problem from the same deficit to someone strong.

In [20]:
if LEAGUE_ID and squad_ids:
    beat = ls.beat_probabilities().merge(
        standings[['entry_id','manager','team_name','total','league_rank']], on='entry_id')
    display(beat.sort_values('league_rank')[
        ['league_rank','manager','team_name','total','p_i_finish_above','exp_gap']])

,league_rank,manager,team_name,total,p_i_finish_above,exp_gap
9,1,Arko Ghosh,Such a Messi guy,378,0.31675,-45.522349
15,2,Dipesh Israni,dips with sauce,374,0.30475,-49.800472
1,3,Shivam Bhandari,Rutter Woke Nonsense,372,0.32775,-42.736829
17,4,Harsh Chandarana,Nico ORELLI,365,0.37775,-30.820260
0,5,Shivam Kotecha,Change Name,362,0.34025,-36.295282
10,6,Shoubhik Bhattacharya,Sho time,362,0.28475,-49.816697
6,7,Farhad Islam,Syldexic Cyclops,357,0.46000,-9.866686
2,7,Devansh Chandgothia,Mounting to the top,357,0.44000,-11.834229
12,9,Avidant Mittal,Show me da Mane,357,0.34475,-38.938200
4,10,Sahil Nathani,KanteGetAnyWorse,356,0.35125,-35.284999


### The moves that most improve where you finish

`dRank` is expected change in your league position from owning a player — negative
is better. It swaps him against your weakest player in that position, so the
counterfactual is a transfer you could actually make. `P(gain)` is how often that
swap improves your finish across simulations; under ~0.55 the model is guessing.

In [21]:
if LEAGUE_ID and squad_ids:
    display(rk.chase_table(board, squad_ids, top=15))

,Player,Team,position,£,Places Gained,dSep,P(gain),ML%,MLFlag,xP6,Ceiling,STS,LTS,Fit,Next 6
0,Saka,ARS,MID,9.6,1.186,0.418,0.568,16.0,DIFF,36.00,51.07,100.0,99.0,100%,LEE(H) NFO(A) EVE(H) LIV(A) HUL(H) NEW(A)
1,Gabriel,ARS,DEF,8.0,1.114,0.524,0.570,0.0,DIFF,32.34,45.00,100.0,100.0,100%,LEE(H) NFO(A) EVE(H) LIV(A) HUL(H) NEW(A)
2,Mbeumo,MUN,MID,7.9,0.813,-0.192,0.454,42.0,DIFF,30.82,45.02,99.0,100.0,100%,TOT(H) LEE(A) BOU(H) CHE(A) AVL(H) LIV(A)
3,Guéhi,MCI,DEF,6.0,0.589,0.877,0.385,0.0,DIFF,25.34,36.70,99.0,99.0,100%,LIV(A) IPS(H) AVL(A) BHA(H) NFO(A) FUL(H)
4,Cherki,MCI,MID,7.8,0.572,0.471,0.382,16.0,DIFF,27.92,41.21,99.0,93.0,100%,LIV(A) IPS(H) AVL(A) BHA(H) NFO(A) FUL(H)
5,Thiago,BRE,FWD,7.8,0.482,1.312,0.363,5.0,DIFF,23.78,35.77,99.0,99.0,100%,AVL(A) LIV(H) HUL(A) NFO(H) BHA(A) EVE(H)
6,Cunha,MUN,MID,7.9,0.476,0.207,0.345,0.0,DIFF,26.68,39.19,98.0,97.0,100%,TOT(H) LEE(A) BOU(H) CHE(A) AVL(H) LIV(A)
7,Havertz,ARS,FWD,7.6,0.473,1.698,0.362,0.0,DIFF,23.70,37.32,91.0,97.0,75%,LEE(H) NFO(A) EVE(H) LIV(A) HUL(H) NEW(A)
8,Raya,ARS,GK,6.1,0.464,0.384,0.348,21.0,DIFF,24.76,33.08,100.0,100.0,100%,LEE(H) NFO(A) EVE(H) LIV(A) HUL(H) NEW(A)
9,Ødegaard,ARS,MID,6.8,0.454,0.053,0.341,11.0,DIFF,26.51,37.49,98.0,92.0,100%,LEE(H) NFO(A) EVE(H) LIV(A) HUL(H) NEW(A)


### The leak: what the field owns that you don't

Ranked by how much their absence costs your finishing position. Nothing on a
per-player sheet flags a player you simply never looked at, and this is the list
that loses mini-leagues quietly.

In [22]:
if LEAGUE_ID and squad_ids:
    display(rk.leak_table(board, squad_ids, top=12))

,Player,Team,position,£,Places Gained,dSep,P(gain),ML%,MLFlag,xP6,Ceiling,STS,LTS,Fit,Next 6
0,Gibbs-White,NFO,MID,8.0,0.352,-0.132,0.303,58.0,COVER,25.09,36.90,95.0,99.0,100%,CRY(A) ARS(H) IPS(A) BRE(A) MCI(H) BOU(A)
1,Calvert-Lewin,LEE,FWD,6.0,0.263,1.192,0.299,53.0,COVER,20.99,31.50,89.0,92.0,100%,ARS(A) MUN(H) SUN(A) BOU(A) TOT(H) CHE(A)
2,Palmer,CHE,MID,9.7,0.217,-0.646,0.270,74.0,COVER,23.36,36.96,95.0,94.0,75%,BOU(H) EVE(A) TOT(H) MUN(H) SUN(A) LEE(H)


### Differentials

Low league ownership, ranked by rank impact rather than raw points. Only read this
as opportunity when you're actually behind — if you're leading, these are how you
lose the lead.

In [23]:
if LEAGUE_ID and squad_ids:
    display(lg.differential_table(board, squad_ids, max_eo=0.35, top=12))

,Player,Team,position,£,ML%,xP6,MLD,Ceiling,pHaul
0,Gabriel,ARS,DEF,8.0,0.0,32.34,32.34,45.00,0.138
1,Saka,ARS,MID,9.6,16.0,36.00,30.32,51.07,0.221
2,Cunha,MUN,MID,7.9,0.0,26.68,26.68,39.19,0.146
3,Guéhi,MCI,DEF,6.0,0.0,25.34,25.34,36.70,0.057
4,Wirtz,LIV,MID,7.3,0.0,25.07,25.07,36.22,0.105
5,Ødegaard,ARS,MID,6.8,11.0,26.51,23.72,37.49,0.133
6,Havertz,ARS,FWD,7.6,0.0,23.70,23.70,37.32,0.101
7,Cherki,MCI,MID,7.8,16.0,27.92,23.51,41.21,0.106
8,Stach,LEE,MID,6.0,0.0,23.33,23.33,33.08,0.044
9,Rúben,MCI,DEF,5.5,0.0,23.16,23.16,32.84,0.040


## 7. Transfer plan (MILP)

Squad, starting XI, captain, transfers, hits and bank are solved *jointly* across
the whole horizon. This is the part a ranked list cannot do: the optimal move is
often a pair — downgrade one player to fund an upgrade elsewhere — and it is
sometimes right to take a hit now for a position you only reach in three weeks.

In [25]:
# per-player per-gameweek expected points, reused by the optimisers below
xpts_all = pd.concat([
    pd.DataFrame({'id': extras['sim']['ids'], 'gw': gw, 'xp': arr.mean(axis=1)})
    for gw, arr in extras['sim']['per_gw'].items()], ignore_index=True)

if squad_ids:
    result = optimise_from_board(board, extras, squad_ids, buy_prices,
                                 bank=BANK, free_transfers=FREE_TRANSFERS,
                                 cfg=cfg, max_hits=1)
    print(result['status'], '| candidate pool:', result['pool_size'])
    display(opt.describe_plan(result, extras['players']))
else:
    print('Fill in MY_SQUAD first.')

AttributeError: type object 'LpVariable' has no attribute 'dicts'

### How confident is the first transfer?

The point estimates are close together most weeks. This is the probability the
incoming player actually outscores the outgoing one over the horizon — under 0.55
means the model is guessing and the transfer is not worth a hit.

In [ ]:
if squad_ids and result['plan'] and result['plan'][0]['in']:
    step = result['plan'][0]
    horizon = list(range(next_gw, next_gw + cfg.long_horizon))
    name = extras['players'].set_index('id')['web_name'].to_dict()
    for o, i in zip(step['out'], step['in']):
        p = sm.transfer_confidence(extras['sim'], o, i, horizon)
        print(f"{name.get(o,o):18s} -> {name.get(i,i):18s}  P(better over {len(horizon)} GWs) = {p:.2f}")

## 8. Captaincy — the upside tail, not the mean

In [ ]:
caps = sm.captaincy_scores(extras['sim'], next_gw).merge(
    board[['id','Player','Team','£','Own%']], on='id')
pool = caps[caps.id.isin(squad_ids)] if squad_ids else caps
show(pool, ['Player','Team','cap_mean','p_haul','p_blank','Own%'], n=8, by='p_haul')

## 9. Best possible squads

What £100m could buy from a blank slate, over the next gameweek, the next three
and the next six. **These ignore transfers** — none of them is a plan you can reach
in one move. They are a ceiling: reading your squad against them shows which
positions are actually costing you points, rather than which players you happen to
like.

In [ ]:
best, comparisons = {}, {}
horizons = {'1gw': [next_gw],
            '3gw': list(range(next_gw, next_gw + 3)),
            '6gw': list(range(next_gw, next_gw + 6))}

for label, gws in horizons.items():
    r = opt.best_squad(xpts_all, extras['players'], gws, budget=100.0, time_limit=60)
    best[label] = r
    comparisons[label] = opt.compare_to_best(r, squad_ids, xpts_all,
                                             extras['players'], gws)
    c = comparisons[label]
    print(f"{label}: best {r['xp']:6.1f} xP for {r['cost']:5.1f}m  |  "
          f"you {c['my_xp']:6.1f}  gap {c['gap']:5.1f}  "
          f"you already own {c['overlap']}/15")

### Where the gap is

Players in the six-gameweek optimum you don't own, and the ones you own that it
doesn't want. This is the least sentimental view of your squad available.

In [ ]:
_nm = extras['players'].set_index('id')['web_name'].to_dict()
print("MISSING (in the best squad, not in yours):")
print('  ' + ', '.join(_nm.get(p, str(p)) for p in comparisons['6gw']['missing']))
print()
print("SURPLUS (in yours, not in the best squad):")
print('  ' + ', '.join(_nm.get(p, str(p)) for p in comparisons['6gw']['surplus']))

## 10. Rivals

Every manager in your league projected on exactly the same basis as you. `vs me`
is the column to read — positive means they are projected to outscore you next
gameweek from their current squad.

In [ ]:
rivals_df = None
if LEAGUE_ID and MY_ENTRY_ID and ls is not None:
    rivals_df = rk.rival_projections(ls, rivals, standings, extras['sim'],
                                     extras['players'], MY_ENTRY_ID, squad_ids,
                                     next_gw)
    display(rivals_df[['league_rank','manager','total','xP1','xP1_vs_me',
                       'captain','xP3','xP6','overlap','p_i_finish_above']])

### Recommended moves

Every row is a transfer you can actually make: affordable against your sale value
plus bank, and legal under the three-per-club cap. Ranked by league places gained.

In [ ]:
moves = None
if LEAGUE_ID and MY_ENTRY_ID:
    _sell = {int(p): opt.sell_price(buy_prices.get(int(p), 0) or 0,
                                    float(extras['players'].set_index('id')
                                          .loc[int(p), 'price']))
             for p in squad_ids}
    # The projected XI, so a bench upgrade is not scored as if it played.
    from fplmodel import lineup as _ln
    _lu = _ln.optimal_lineup(extras['sim'], next_gw, squad_ids,
                             dict(zip(board.id.astype(int),
                                      board.element_type.astype(int))),
                             p_appear=dict(zip(board.id.astype(int),
                                               board.p_appear.fillna(0))))
    _xi = {int(x) for x in (_lu.get('xi') or [])}
    moves = rk.transfer_recommendations(board, squad_ids, _sell, BANK, top=20,
                                        xi_ids=_xi)

# The decision itself: the best COMBINATION of transfers, scored on the eleven
# you would field each gameweek. `moves` above is still the browse-able list of
# one-for-one swaps; this is the answer.
PLANS = None
if squad_ids:
    _hz = sorted(g for g in extras['sim']['per_gw'] if g >= next_gw)[:6]
    _xpw = {g: {int(i): float(extras['sim']['per_gw'][g][k].mean())
                for k, i in enumerate(extras['sim']['ids'])} for g in _hz}
    _meta = board[['id', 'Player', 'Team', 'team', 'element_type', '£']]
    _sellp = {int(p): opt.sell_price(buy_prices.get(int(p), 0) or 0,
                                     float(extras['players'].set_index('id')
                                           .loc[int(p), 'price']))
              for p in squad_ids}
    _plans = opt.transfer_plans_weekly(
        _xpw, _meta, squad_ids, _sellp, BANK,
        free_transfers=FREE_TRANSFERS, max_hit=MAX_HIT,
        # your players' own appearance odds, not a league average
        p_appear=dict(zip(board.id.astype(int),
                          board.p_appear.fillna(1.0).astype(float))))
    PLANS = sht.plan_values(_plans, _hz, free_transfers=FREE_TRANSFERS)
    for _p in _plans:
        _o = ' + '.join(f'{x[0]}' for x in _p['out']) or '-'
        _i = ' + '.join(f'{x[0]}' for x in _p['in']) or '-'
        print(f"{_p['n']}T alt{_p.get('rank', 1)}  hit -{int(_p['hit'])}  "
              f"best-XI xP {_p.get('gross_xi', _p['gross']):.1f}  "
              f"XI gain {_p.get('gain_xi', _p['gain']):+.1f} "
              f"(+bench {_p['gain']:+.1f}, after curse "
              f"{_p.get('gain_discounted', _p['gain']):+.1f})   {_o} -> {_i}")

    # The same question asked properly: a free transfer arrives EVERY week, so
    # the right first move is the first step of a six-week path, not the best
    # thing you can do if the squad then stands still. This MILP has been in
    # the pipeline since 1.0 and was never shown.
    try:
        _path = optimise_from_board(board, extras, squad_ids, buy_prices,
                                       BANK, FREE_TRANSFERS, cfg,
                                       max_hits=MAX_HIT // 4)
        _nm = dict(zip(board.id.astype(int), board.Player))
        _pr = dict(zip(board.id.astype(int), board.price.astype(float)))
        PATH = sht.path_values(_path.get('plan'), _nm, _pr, first_gw=next_gw)
        PLANS = PLANS + [[], []] + PATH
        print()
        for _w in _path.get('plan', []):
            _o = ' + '.join(_nm.get(i, i) for i in _w['out']) or '-'
            _i = ' + '.join(_nm.get(i, i) for i in _w['in']) or '-'
            print(f"GW{_w['gw']}  {_o} -> {_i}   bank {_w['bank']:.1f}  "
                  f"C:{_nm.get(_w['captain'], '')}")
    except Exception as _exc:
        print(f"six-week path unavailable ({_exc})")

    display(moves)

## 11. Lineup and captaincy

Points you can lose without transferring anything: a benched returner, a bench in
the wrong order, or an armband that matches the field.

`capEO` is the share of your league captaining that player. `dRank` is the expected
change in your finishing position from picking him over the field's consensus —
**negative means it helps**, and if every alternative is positive, the template
captain is genuinely correct and getting clever costs you places.

In [ ]:
from fplmodel import lineup as ln

_pa = dict(zip(board.id.astype(int), board.p_appear.fillna(0)))
_pos = dict(zip(board.id.astype(int), board.element_type.astype(int)))
lu = ln.optimal_lineup(extras['sim'], next_gw, squad_ids, _pos, p_appear=_pa)
_nm2 = extras['players'].set_index('id')['web_name'].to_dict()

print(f"formation {lu['formation']} | projected XI xP {lu['xp']:.1f}")
print("XI     :", ', '.join(_nm2[p] for p in lu['xi']))
print("CAPTAIN:", _nm2.get(lu['captain']), '| VICE:', _nm2.get(lu['vice']))
print("BENCH  :", ' > '.join(_nm2[p] for p in lu['bench_gk'] + lu['bench']))

# Write the squad and this XI into the drift log, against the SAME run
# `run_weekly` recorded a moment ago. This is the only record that will ever
# exist of what the model projected for the team actually fielded -- FPL will
# tell us the score afterwards, but never what we expected.
try:
    from fplmodel import tracking as tk
    if extras.get('tracking'):
        tk.record_squad(cfg, extras['tracking'], squad_ids,
                        entry_id=MY_ENTRY_ID, lineup=lu)
except Exception as _e:
    print('tracking: squad not recorded --', type(_e).__name__, _e)

capt, rival_caps, tc = None, None, {}
if LEAGUE_ID and MY_ENTRY_ID and ls is not None:
    capt = ln.captain_options(extras['sim'], next_gw, lu['xi'], extras['players'],
                              rival_captains=ln.captains_by_entry(picks, MY_ENTRY_ID),
                              ls=ls, squad_ids=squad_ids, positions=_pos, top=8)
    rival_caps = ln.rival_captain_counts(picks, extras['players'], MY_ENTRY_ID)
    tc = ln.triple_captain_value(extras['sim'], next_gw, lu['captain'])
    display(show(capt, ['Player','Team','xP','pHaul','pBlank','Ceiling',
                        'capEO','Places Gained','isFieldPick']))

### What each remaining chip is worth

The Wildcard's value is the gap between your squad and the optimum. When that gap
is large it dwarfs any single transfer, and comparing a one-move decision against
it is the wrong comparison.

In [ ]:
chips_df = ln.chip_value(best, comparisons, mtm.remaining_chips(team.chips_used))
display(chips_df)
if tc:
    print('Triple Captain on this armband would add', tc['extra_mean'],
          'points on average,', tc['extra_p90'], 'at the 90th percentile')

## 12. Team form

Form is real, but not the way it feels. Results-based form is mostly finishing
variance and regresses fast; **underlying** form — a genuine shift in expected
goals — persists, because it reflects actual squad or system change.

`Drift` is how far the model has moved a team from last season's baseline.
`Luck` is goals minus expected goals: **positive means they have been finishing
above their chances and are due to regress**, which is the usual explanation for
a team "beating sides above them".

Recent matches already weigh more via exponential decay with a 213-day half-life
— the value optimised on five seasons of Premier League data. Shorter half-lives
chase form and forecast *worse*.

In [ ]:
from fplmodel import form as fmod

_pa, _pd = extras['prior'] if extras['prior'] else ({}, {})
teamform = fmod.team_form(extras['dc'], _pa, _pd, extras['matches'], extras['teams'])
teamform['verdict'] = teamform.apply(fmod.form_verdict, axis=1)
display(teamform[['short','now','base','delta','n','gd90','xgd90','luck','verdict']])

### Is form doing anything at all?

If the ratings barely move between a 90-day and a 400-day half-life, then form
is not influencing your projections and no amount of tuning will change that.
Worth checking once rather than assuming.

In [ ]:
try:
    display(fmod.sensitivity(cfg, extras['matches'], extras['teams'],
                             extras['prior'], half_lives=(90, 213, 400)).head(10))
except Exception as e:
    print('sensitivity needs a few more matches:', e)

## 13. Live gameweek

Run this **while the matches are on**. It does not re-run the model: the
projection for a fixture that has not kicked off does not change while other
games are being played, so a refresh is two API calls and a couple of seconds
against the distribution the last full run cached.

Three bands on the tab, because the numbers mean three different things.
**Banked** is settled. **In play** includes provisional bonus, which FPL has
not awarded yet. **To come** is the only part that is a forecast.

Autosubs and the vice-captain are applied, but only once a player's match is
over and he did not appear — never while it is still running and he might yet
come on.

In [ ]:
from fplmodel.pipeline import refresh_live

# One call. It fetches the league's CURRENT squads itself -- rivals' picks go
# public at the deadline, so during a live gameweek they exist, and they are
# the only ones that describe what is actually on the pitch. Every other tab
# uses the LAST COMPLETED gameweek's squads, which is why this cannot reuse
# the `picks` frame the mini-league section built.
LIVE_PAYLOAD = None
if LEAGUE_ID and MY_ENTRY_ID:
    LIVE_PAYLOAD = refresh_live(cfg, league_id=LEAGUE_ID, my_entry=MY_ENTRY_ID)

if LIVE_PAYLOAD:
    _t = LIVE_PAYLOAD['table']
    display(_t[['was', 'manager', 'chip', 'banked', 'live', 'prov_bonus',
                'to_come', 'to_play', 'projected', 'floor', 'ceiling',
                'now', 'rank_change', 'p_above']])
    print('The Live tab is written by the next cell, with the workbook.')
elif not LEAGUE_ID:
    print('Set league_id in fpl_settings.json -- the Live tab is a league view.')
else:
    print('No gameweek in progress, so there is nothing live to show.')

## 14. Write the Google Sheet

Creates GK / DEF / MID / FWD / My Team / League / Best Teams / Rivals / Targets / Lineup / Teams / Accuracy / Glossary, plus a hidden `_squad` tab.

**Glossary** defines every metric and what decision it should drive.

The `_squad` tab is yours — the weekly run never writes to it. Your `In`, `Buy£`
and `BuyGW` entries live there and the position tabs VLOOKUP into them, so the
refresh can clear and rewrite everything else without touching your data.

In [ ]:
from google.colab import auth
import gspread
from google.auth import default
auth.authenticate_user()
gc = gspread.authorize(default()[0])

frames = {p: sht.build_position_frame(board, p, cfg) for p in ('GK','DEF','MID','FWD')}

seed = None
if squad_ids:
    seed = pd.DataFrame({
        'id': squad_ids,
        'buy_price': [buy_prices.get(int(p), np.nan) for p in squad_ids],
        'buy_gw':    [buy_gws.get(int(p), 1) for p in squad_ids],
        'name':      [board.set_index('id')['Player'].get(int(p), '') for p in squad_ids],
    })

summary = sht.squad_summary(board, squad_ids, extras['proj'], next_gw,
                            bank=BANK, chips=CHIPS_LEFT,
                            buy_prices=buy_prices) if squad_ids else {}
if ctx:
    _sep = ls.separation() if ls is not None else {}
    summary.update({k: round(v, 3) for k, v in _sep.items()})
    summary.update({'my_rank': ctx.get('my_rank'), 'n_teams': ctx.get('n_teams'),
                    'points_behind_leader': ctx.get('points_behind_leader'),
                    'league_stance': ctx.get('stance'), 'league_read': ctx.get('read'),
                    'ml_as_of_gw': next_gw - 1})

league_payload = None
if LEAGUE_ID and squad_ids and ls is not None:
    league_payload = rk.build_league_payload(standings, ls, board, squad_ids,
                                             MY_ENTRY_ID, ctx, as_of_gw=next_gw - 1)

# Is a gameweek in progress? If so the model sheets stay exactly as the last
# COMPLETE gameweek left them and only the Live tab moves. Re-running the model
# on a half-played gameweek rewrites every ranking in favour of whoever kicked
# off first, which is worse than not re-running at all.
# The model has learned only from COMPLETE gameweeks -- gw_history stops at
# extras['data_gw'] -- so these projections are as of that gameweek's end no
# matter when you run. What the freeze prevents is re-running them a second
# time on a half-played week, which would rewrite every ranking in favour of
# whoever kicked off first.
#
# So the rule is "write once per gameweek, then hold", not "never write".
import gspread as _gspread
try:
    _ss = gc.open(cfg.sheet_name)
except _gspread.SpreadsheetNotFound:
    _ss = gc.create(cfg.sheet_name)

GW_STATE = extras['fpl'].gameweek_state(next_gw)
SETTLING = extras['fpl'].awaiting_settlement(next_gw)
ALREADY = sht.model_already_written_for(_ss, next_gw)
MODEL_FROZEN = (GW_STATE == 'live') and ALREADY

# "GW3 is live" reads as "matches are being played". When every match has
# finished and FPL simply has not set data_checked, that wording is actively
# misleading -- it sends you to the API to find out whether the tool is stale.
if SETTLING:
    print(f"GW{next_gw}: all matches PLAYED, but FPL has not settled the "
          f"gameweek yet (data_checked is still false).")
    print(f"  Nothing to project: GW{next_gw}'s fixtures are over, and the "
          f"model may not learn from it until FPL signs the stats off --")
    print(f"  the dubious goals panel still moves goals and assists for a day "
          f"or two after the whistle.")
    print(f"  Re-run once it settles and you will get GW{next_gw + 1} off "
          f"{next_gw} gameweeks of data. The Live tab below still updates.")
elif GW_STATE == 'live':
    print(f"GW{next_gw} is IN PROGRESS; model data is complete through "
          f"GW{extras['data_gw']}.")
else:
    print(f"GW{next_gw} is {GW_STATE}; model data is complete through "
          f"GW{extras['data_gw']}.")
print('Model sheets FROZEN -- already written for this gameweek, only the '
      'Live tab will move.' if MODEL_FROZEN
      else 'Writing every sheet from complete-gameweek data.')

ss = sht.write_workbook(frames, seed, summary, cfg, spreadsheet=_ss,
                        league=league_payload, squad_ids=squad_ids,
                        live=LIVE_PAYLOAD,
                        model_frozen=MODEL_FROZEN, model_gw=next_gw,
                        gws_played=extras['data_gw'])

if not MODEL_FROZEN:
  sht.write_extra_tabs(
    ss, best, comparisons, board, squad_ids,
    rivals_df, next_gw - 1,
    rk.chase_table(board, squad_ids, top=20) if LEAGUE_ID else None,
    rk.leak_table(board, squad_ids, top=15) if LEAGUE_ID else None,
    moves, ctx, plans=PLANS)

  # Once the deadline has gone, read back what was actually submitted so the
  # tab reconciles rather than advises. `myteam.fetch_picks` is the record.
  SUBMITTED = None
  if MY_ENTRY_ID and extras['fpl'].picks_public(next_gw):
      try:
          _pk = mtm.fetch_picks(MY_ENTRY_ID, next_gw, cfg)
          _rows = _pk.get('picks', [])
          _m = {int(r['element']): float(r.get('multiplier', 1)) for r in _rows}
          _cap = [int(r['element']) for r in _rows if r.get('is_captain')]
          SUBMITTED = {
              'xi': [p for p, m in _m.items() if m >= 1],
              'captain': _cap[0] if _cap else None,
              'multipliers': _m,
              'chip': _pk.get('active_chip') or '-',
          }
          print(f"submitted GW{next_gw} team read back from FPL"
                + (f" (chip: {SUBMITTED['chip']})"
                   if SUBMITTED['chip'] not in ('-', None, '') else ''))
      except Exception as _exc:
          print(f"could not read back the submitted team: {_exc!r}")

  sht.write_lineup_tab(ss, sht.lineup_values(lu, capt, board, rival_caps,
                                             chips_df, tc, next_gw,
                                             submitted=SUBMITTED))
  sht.write_teams_tab(ss, sht.teams_values(teamform, cfg.dc_half_life_days))

  from fplmodel import tracking as tk
  _acc = tk.drift_summary(cfg)
  _last = int(_acc['gw'].max()) if len(_acc) else None
  sht.write_accuracy_tab(ss, sht.accuracy_values(
      _acc,
      tk.worst_misses(cfg, gw=_last, n=15) if _last else None,
      tk.player_bias(cfg, min_gws=3, n=25)))
print('written:', ss.url)

## 14b. Drift — how far off was the model last time?

Every other section says what the model *thinks*. This one says how far off it was,
which is the only thing that can tell you whether to believe the rest.

The projection is written down on every run and scored against **the last run before
the deadline**. That restriction is the whole point: a run made on Sunday night has
already seen half the football, so scoring it would flatter the model for nothing.
Weeks scored against a post-deadline run are marked `Clean = NO` and left out of the
running averages rather than dropped.

Read `Bias to date` and `Corr to date`, not the single-week columns. One gameweek is
ten matches and mostly variance. Bias drifting away from zero is a calibration
problem and is fixable; a rank correlation stuck near zero means the model is not
picking players, which is the failure that actually costs transfers.

This is also the only genuinely out-of-sample measurement the project has — the
backtest runs on the same season most of the model's constants were fitted against.

In [ ]:
from fplmodel import tracking as tk

_drift = tk.drift_summary(cfg)
if not len(_drift):
    print('Nothing scored yet. The first row appears once a gameweek with a '
          'recorded projection has settled -- so from the week after this one.')
else:
    display(_drift)
    _last = int(_drift['gw'].max())
    print(f'\nbiggest misses, GW{_last}:')
    display(tk.worst_misses(cfg, gw=_last, n=10))
    _pb = tk.player_bias(cfg, min_gws=3, n=15)
    if len(_pb):
        print('\npersistently wrong, across gameweeks:')
        display(_pb)

## 15. Validation — does this beat doing nothing?

Walk-forward, never using future data, against four baselines. Losing to `ep_next`
or `last4` means the model is not worth running yet.

Two gameweeks in there is almost nothing to validate on. Run this properly around
GW8–10, and again after the January window.

In [ ]:
history = extras['history']
completed = sorted(history.gw.unique()) if len(history) else []

if len(completed) >= 4:
    def pipeline_fn(gw):
        b = val.build_baselines(extras['players'], history,
                                board[['id','xmins']].rename(columns={'xmins':'xmins'}),
                                target_gw=gw)
        b['xp_model'] = board.set_index('id')['xP2'].reindex(b['id']).to_numpy() / 2
        return b

    res = val.walk_forward(pipeline_fn, history,
                           first_gw=completed[2], last_gw=completed[-1])
    display(res)
    print(val.verdict(res))
else:
    print(f'Only {len(completed)} completed gameweeks. '
          'Come back around GW8 — validating on two weeks tells you nothing.')

## 16. Publish to the app

Pushes this week's numbers, and any new player photos, to
`github.com/sharad-25/fpl-data`. The APK reads that repo: one artefact a week,
identical for every user, no backend and nothing personal in it — squads,
leagues and live scores are fetched from FPL's own API on the device.

Needs `GH_TOKEN` in Colab Secrets (left sidebar → key icon → add it **and**
switch Notebook access on). Skips itself cleanly if the secret is missing, so
a normal run is not held hostage to it.

In [ ]:
from pathlib import Path
GH_USER, GH_REPO = 'sharad-25', 'fpl-data'
EMAIL = 'sharad.shrivastava00@gmail.com'
OUT = Path('/content/fpl-data')

import shutil, subprocess
try:
    from google.colab import userdata
    TOKEN = userdata.get('GH_TOKEN')
except Exception:
    TOKEN = None

if not TOKEN:
    print('GH_TOKEN not set — skipping the publish step. '
          'Colab sidebar -> key icon -> add the secret AND turn Notebook access on.')
else:
    from fplmodel import export_app, export_assets

    def git(*args, cwd=OUT):
        r = subprocess.run(['git', *args], cwd=str(cwd), capture_output=True, text=True)
        if r.returncode and 'nothing to commit' not in (r.stdout + r.stderr):
            raise RuntimeError(f"git {' '.join(args)}\n{r.stdout}{r.stderr}")
        return r.stdout.strip()

    # A fresh clone every week. Colab wipes /content between sessions anyway,
    # and a fresh clone can never push a stale branch over a good one.
    shutil.rmtree(OUT, ignore_errors=True)
    # Shallow: by the end of a season this repo holds ~80 MB of weekly history
    # and none of it is needed to add one commit. The working tree is complete
    # either way, which is what `export_assets` checks to decide what is
    # already published.
    subprocess.run(['git', 'clone', '--quiet', '--depth', '1',
                    f'https://{TOKEN}@github.com/{GH_USER}/{GH_REPO}.git', str(OUT)],
                   check=True)
    git('config', 'user.email', EMAIL)
    git('config', 'user.name', 'fpl-model')

    manifest = export_app.write(board, extras, cfg, OUT)

    # Faces, badges and kits. Incremental -- it writes only what is missing, so
    # the first run costs a few megabytes and every later one costs whatever a
    # new signing weighs. The Premier League CDN rate-limits a burst of several
    # hundred requests and simply refuses some; that is not an error, and the
    # next run collects whatever it missed.
    assets = export_assets.write(extras['players'], extras['teams'], OUT)

    git('add', '-A')
    out = git('commit', '-m', f"GW{manifest['gw']} \u00b7 model {manifest['model']}")
    if 'nothing to commit' in out:
        print('no change since the last push — nothing to do')
    else:
        git('push')
        kb = sum(f['bytes'] for f in manifest['files'].values()) // 1024
        print(f"\nGW{manifest['gw']} published \u00b7 model {manifest['model']} \u00b7 {kb} KB")
        for name, f in sorted(manifest['files'].items()):
            print(f"   {name:<22}{f['bytes']//1024:>6} KB   {f['sha']}")
        if assets.get('fetched'):
            print(f"   {assets['fetched']} new images (+{assets['bytes']//1024} KB), "
                  f"{assets.get('total_bytes', 0)/1024/1024:.1f} MB of images in the repo")
    print(f"\nhttps://raw.githubusercontent.com/{GH_USER}/{GH_REPO}/main/manifest.json")

---

### Known limits of v1

* **Shot volume is inferred from xG**, not measured — the FPL feed has no shot counts.
  Joining Understat gives real `shots` and `npxG` and makes the volume/quality split
  meaningful rather than cosmetic.
* **`SiB/90` is a proxy** (shots per 90) and `BC/90` / `PSxG−GA` are blank for the
  same reason.
* **Penalties are folded into xG**, not modelled separately, because the FPL
  `expected_goals` field already includes them. Set `separate_penalties=True` in
  `build_parameters` only once you have true npxG, or you will count them twice.
* **Attacking returns are sampled independently** within a team. Clean sheets are
  correctly shared, which is the correlation that matters most for squad variance,
  but two forwards from the same side are treated as independent.
* **Bonus ties** are resolved by ranking rather than FPL's duplicate-award rule,
  which slightly understates bonus in low-scoring matches.

---

## Verify (run once, share the output)

Hits the real endpoints with your ids and prints a compact report. Everything the
model relies on is checked here: squad import, buy-price reconstruction, league
picks, effective ownership, fixtures and odds coverage.

In [ ]:
import json, traceback
rep = {}

def _try(label, fn):
    try:
        rep[label] = fn()
    except Exception as e:
        rep[label] = f'ERROR {type(e).__name__}: {e}'

_try('version', lambda: fplmodel.__version__)
_try('next_gw', lambda: next_gw)
_try('players_scored', lambda: len(board))
_try('history_rows', lambda: len(extras['history']))

_try('squad', lambda: {
    'n': len(squad_ids), 'bank': BANK, 'ft': FREE_TRANSFERS,
    'chips_used': team.chips_used, 'as_of_gw': team.as_of_gw,
    'buy_prices_found': sum(1 for p in squad_ids if p in buy_prices),
    'warnings': team.warnings,
    'players': sorted(board.set_index('id').loc[squad_ids, 'Player'].tolist()),
})

_try('fixtures', lambda: {
    'rows': len(extras['proj']),
    'from_odds': int((extras['proj']['source'] == 'odds').sum()),
    'teams': int(extras['proj']['team'].nunique()),
})
_try('shrinkage_k', lambda: {k: round(v) for k, v in extras['shrinkage_k'].items()})

_try('league', lambda: {
    'teams': len(standings),
    'my_rank': ctx.get('my_rank'), 'behind': ctx.get('points_behind_leader'),
    'stance': ctx.get('stance'),
    'picks_rows': len(picks), 'rivals': len(rivals),
    'eo_players': len(eo),
    'top_owned': eo.nlargest(5, 'ml_eo')[['id', 'ml_eo']].to_dict('records'),
})
_try('rank_outlook', lambda: {k: round(v, 3) for k, v in outlook.items()})
_try('best_squads', lambda: {k: {'cost': v['cost'], 'xp': v['xp'],
                                 'status': v['status']} for k, v in best.items()})
_try('gaps', lambda: {k: v['gap'] for k, v in comparisons.items()})
_try('moves', lambda: len(moves) if moves is not None else 0)
_try('columns_blank', lambda: sorted(
    c for c in sht.COLUMNS if c in board.columns and board[c].isna().all()))

print(json.dumps(rep, indent=2, default=str))